# Quadmesh 3B — Training Commands (no `--detach`)

Plain commands, no `--detach`. Each `modal run` cell blocks until that step finishes and streams its
own log straight into the cell output, which is saved with the notebook (`Ctrl+S`) - that is the
command+log record, no wrapper code needed.

**Real tradeoff to know before running this version:** without `--detach`, if the notebook's own
execution environment has a shorter timeout than the job takes (this happened earlier here, on just
the ~3-minute tokenizer build), the cell will report a timeout/error even though the job may keep
running on Modal in the background. For the training cells specifically (`shared_pretrain`, which can
run for days-to-weeks), this is very likely to happen - the cell will almost certainly appear to fail
locally while the real job continues on Modal. Always check https://modal.com/apps for the real status
if a cell here looks like it failed - that dashboard is the source of truth, not this notebook.

## 1. Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -r requirements-local-agent.txt

In [ ]:
!python tests/run_offline_tests.py

## 2. Accounts

In [ ]:
!huggingface-cli login

In [ ]:
!modal setup

In [ ]:
!modal secret create huggingface-secret HF_TOKEN=<YOUR-TOKEN>

## 3. Tokenizer + review queue (once)

Using the Python SDK directly with `modal.enable_output()` here instead of the
shell `!modal run` - `build_tokenizer` is a plain `@app.function`, which is exactly the case where Jupyter's
`!` subprocess wrapper does not reliably stream remote logs back without this context manager. This should
give real, live line-by-line output in the cell, matching the dashboard.

In [ ]:
import modal_app, modal

with modal.enable_output():
    with modal_app.app.run():
        modal_app.build_tokenizer.remote()

Text2CAD is a gated dataset - if the log shows `Finished Text2CAD stream (0 docs)`, accept its
license at https://huggingface.co/datasets/SadilKhan/Text2CAD with the same HF account as your token,
then re-run the cell above.

In [ ]:
!modal run modal_app.py::shared_review_queue

In [ ]:
!modal volume get quadmesh-ckpts /data/review_queue.jsonl review_queue.jsonl

Run this one in a separate terminal, not the notebook - it's interactive:

`python -m quadmesh.pipeline.review_queue review_queue.jsonl`

In [ ]:
!modal volume put quadmesh-ckpts review_queue.jsonl /data/review_queue.jsonl --force

## 4. Train — chinchilla budget first (cheap validation pass)

No `--detach`: this cell will try to stream the full pretrain run's log live and block until it's done.
Given the real run length (days), expect this cell to time out locally well before training actually
finishes - check the dashboard for real status, and re-run this exact cell to resume from the last
checkpoint if it does time out or get interrupted.

In [ ]:
!modal run modal_app.py::shared_pretrain --tier 3B --budget chinchilla --mb 16 --accum 16 --grad_ckpt 0 --workers 12

Wait for the cell above to actually reach `done=true` (check the dashboard) before running SFT below - SFT loads the pretrain checkpoint as its starting point.

In [ ]:
!modal run modal_app.py::shared_sft --tier 3B --tokens 2000000000 --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run modal_app.py::shared_eval --tier 3B --n 100

## 5. Download + build the final report once training shows `done: true`

In [ ]:
!modal volume get quadmesh-ckpts /Quadmesh-3B ./Quadmesh-3B

In [ ]:
!python -m quadmesh.pipeline.training_report --tier 3B --ckpt-dir ./Quadmesh-3B --out training_report.md

In [ ]:
!type training_report.md

## 6. Only if chinchilla's results justify it — full 4x run

Same commands, `--budget 4x` instead of `chinchilla`. Overwrites the chinchilla checkpoint (same fixed
filename on the volume) - a fresh longer run, not additive.

In [ ]:
!modal run modal_app.py::shared_pretrain --tier 3B --budget 4x --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run modal_app.py::shared_sft --tier 3B --tokens 2000000000 --mb 16 --accum 16 --grad_ckpt 0 --workers 12

In [ ]:
!modal run modal_app.py::shared_eval --tier 3B --n 100